# 🐝 YOLO11n ile arı sayma: Colab'da eğitim + sayma eşiği seçimi
**Edge AI serisi, Bölüm 7.** Google Colab'da (**T4 GPU**) çalışır. Çıktı (`best.pt`) Raspberry Pi 5'e götürülür, NCNN'ye çevrilir ve 7/24 çalışan bir sayma sistemine dönüşür.

**Başlamadan:**
1. `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`
2. Sol menü 🔑 **Secrets** → `ROBOFLOW_API_KEY` adıyla Roboflow anahtarını ekle → **Notebook access** açık

| Hücre | İş |
|---|---|
| 1 | Kurulum, GPU ve anahtar kontrolü |
| 2 | Veri setini indir (Roboflow, v1, 16.228 resim) |
| 3 | Veri setini SAYILARLA tanı: boş resim oranı, arı/resim, kutu boyutları |
| 4 | Eğitim: YOLO11n, imgsz=640, fraction=0.25, cache=ram, Drive'a kayıt |
| 5 | Grafikler: results, confusion matrix, labels |
| 6 | Sayma için eşik seçimi: MAE ve bias (test kümesi) |
| 7 | best.pt + test kümesini indir |

## 1) Kurulum ve kontroller
Beklenen: `CUDA:0 (Tesla T4)` ve `API anahtarı bulundu ✅`. Anahtarın kendisi **asla** yazdırılmaz, sadece uzunluğu.

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

from google.colab import userdata
key = userdata.get("ROBOFLOW_API_KEY")
print("API anahtarı bulundu ✅" if key else "❌ Anahtar yok")
print("Anahtar uzunluğu:", len(key))

## 2) Veri setini indir
Roboflow Universe → `marcelo-rovai-riila / bees_on_hive_landing_boards`, sürüm 1. Ön işleme: **Stretch to 640×640**. Augmentation: sadece train ×3 (döndürme ±15°, parlaklık ±15%, pozlama ±10%).

In [ ]:
%cd /content
from roboflow import Roboflow
rf = Roboflow(api_key=key)
project = rf.workspace("marcelo-rovai-riila").project("bees_on_hive_landing_boards")
dataset = project.version(1).download("yolov11")
DS = dataset.location
print("Veri seti:", DS)

## 3) Veri setini sayılarla tanı
- 6.762 orijinal → %70/%20/%10 → **sadece train ×3** = 14.199 + 1.353 + 676
- Kutu boyutu: genişlik ortanca ~19 px, yükseklik ~40 px (stretch: yatay ÷3.0, dikey ÷1.69) → **imgsz = 640** (en dar %10 ≈ 14 px; 320'de bir P3 hücresinden küçük kalırdı)

In [ ]:
import glob
import numpy as np

print(f"{'Küme':6s} {'resim':>6s} {'arı':>7s} {'boş':>11s} {'ort/resim':>10s} {'en fazla':>9s}")
for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{DS}/{split}/images/*")
    lbls = glob.glob(f"{DS}/{split}/labels/*.txt")
    counts = np.array([sum(1 for line in open(f) if line.strip()) for f in lbls])
    bos = int((counts == 0).sum())
    print(f"{split:6s} {len(imgs):6d} {counts.sum():7d} {bos:5d} (%{100 * bos / len(lbls):4.1f}) "
          f"{counts.mean():10.1f} {counts.max():9d}")

ws, hs = [], []
for f in glob.glob(f"{DS}/valid/labels/*.txt"):
    for line in open(f):
        p = line.split()
        if len(p) == 5:
            ws.append(float(p[3]) * 640); hs.append(float(p[4]) * 640)
ws, hs = np.array(ws), np.array(hs)
print("\nArı kutusu boyutu (piksel, 640×640):")
print(f"  genişlik : ortanca {np.median(ws):4.0f} | en küçük %10: {np.percentile(ws, 10):3.0f} | en büyük %10: {np.percentile(ws, 90):3.0f}")
print(f"  yükseklik: ortanca {np.median(hs):4.0f} | en küçük %10: {np.percentile(hs, 10):3.0f} | en büyük %10: {np.percentile(hs, 90):3.0f}")

## 4) Eğitim
- **fraction=0.25**: Tüm veriyle kitapta 2.7 saat; %25 ile ~18 dakika. ⚠️ `fraction` RASTGELE değil, alfabetik İLK %25'i alır → bizde boş resim oranı %7.4'ten %2.3'e düştü, sahneler daha sakin (6.3 arı/resim). Daha iyisi: rastgele/tabakalı alt küme ya da tüm veri.
- **cache="ram"**: Colab'da sadece 2 CPU var; JPEG açma yükünü kaldırır (~5.6 GB RAM).
- **project = Drive**: Colab oturumu koparsa model kaybolmaz.

In [ ]:
from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/gdrive")
SAVE = "/content/gdrive/MyDrive/edge_ai/bees"

open(f"{DS}/data.yaml", "w").write(f"""path: {DS}
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['bee']
""")

model = YOLO("yolo11n.pt")
results = model.train(data=f"{DS}/data.yaml", epochs=20, imgsz=640, fraction=0.25,
                      cache="ram", project=SAVE, name="bees_640_f25", plots=True)

## 5) Grafikler
Confusion matrix varsayılan **conf=0.001** ile çizilir: background sütunundaki on binlerce kutu düşük güvenli çöp kutulardır; çalışma eşiğinde değerlendirme 6. hücrede.

In [ ]:
from IPython.display import Image, display
R = "/content/gdrive/MyDrive/edge_ai/bees/bees_640_f25"     # tam yol: oturum sıfırlansa da çalışır
display(Image(f"{R}/results.png", width=900))
display(Image(f"{R}/confusion_matrix.png", width=450))
display(Image(f"{R}/labels.jpg", width=600))

## 6) Sayma için eşik seçimi
Sayma sorusu mAP değil: **MAE** (resim başına ortalama mutlak hata) ve **bias** (sistematik fazla/eksik sayma).
- Model BİR kez conf=0.05 ile çalıştırılır, eşikler sonradan uygulanır (NMS yalnızca daha yüksek skorlu kutuyla çakışanı sildiği için sonuç aynıdır).
- ⚠️ Ultralytics'e **liste** verme: Liste tek bir batch olur → 676 resim aynı anda GPU'ya → OOM. **Klasör** ver.
- ⚠️ Doğrusu eşiği **valid** kümesinde seçmek, **test**'te raporlamaktır (burada öğretim amaçlı test kullanıldı).

Bizim sonuç: conf=0.3 → MAE 1.22, bias −0.14 (toplam −%1.8).

In [ ]:
import gc, os
import torch
gc.collect(); torch.cuda.empty_cache()

best = YOLO(f"{R}/weights/best.pt")

def gercek_sayi(img):
    lbl = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    return sum(1 for line in open(lbl) if line.strip()) if os.path.exists(lbl) else 0

gt, skorlar = [], []
for r in best.predict(f"{DS}/test/images", imgsz=640, conf=0.05, iou=0.7, verbose=False, stream=True):
    gt.append(gercek_sayi(r.path))
    skorlar.append(r.boxes.conf.cpu().numpy())
gt = np.array(gt)

print(f"{len(gt)} test resmi | gerçek toplam: {gt.sum()} arı\n")
print(f"{'conf':>5s} {'tahmin':>7s} {'MAE':>6s} {'bias':>7s} {'tam doğru':>10s}")
for c in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    tahmin = np.array([(s >= c).sum() for s in skorlar])
    hata = tahmin - gt
    print(f"{c:5.1f} {tahmin.sum():7d} {np.abs(hata).mean():6.2f} {hata.mean():+7.2f} {100 * (hata == 0).mean():9.1f}%")

## 7) best.pt ve test kümesini indir
Windows'ta: `scp best.pt pi@<PI_IP>:Documents/YOLO/models/bees_640_yolo11n.pt`
Pi'de: `yolo export model=models/bees_640_yolo11n.pt format=ncnn imgsz=640` → çıkış `(1, 5, 8400)`

In [ ]:
!cd {DS} && zip -r -q /content/bees_test.zip test
!cp /content/bees_test.zip {R}/
from google.colab import files
files.download(f"{R}/weights/best.pt")
files.download("/content/bees_test.zip")